> **교육 목표** 학습한 모델을 현장 사람이 쓸 수 있도록 저장하고 Streamlit 예측 앱으로 만듭니다.
>
> **핵심 내용** Pipeline 학습 → `joblib`으로 모델과 메타정보(`ranges`, `columns`) 저장 → 다시 불러와 확인 → 앱 작성과 실행

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D4_05 강도 예측 앱 만들기

> **오늘 질문:** 노트북에서 만든 모델을 현장 사람이 쓰려면 무엇이 필요할까?

**구조:** 노트북에서 학습 → `joblib`으로 저장 → 앱에서 불러와 예측만

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

cc = pd.read_csv("../../data/concrete.csv")
X = cc.drop(columns="strength")
y = cc["strength"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Pipeline 학습 ✍️

전처리와 모델을 Pipeline 하나로 묶으면, 저장·불러오기를 한 번에 할 수 있고 전처리 기준이 어긋나지 않습니다.

> 변수 이름: `model`, `mae`

In [ ]:
# 힌트: make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=200, random_state=42))

## 2. 모델과 메타정보를 함께 저장 ✍️

| 함께 저장할 것 | 쓰임 |
|---|---|
| `model` | 예측 |
| `columns` | 입력 컬럼 이름과 순서 |
| `ranges` | 학습 범위(최소·최대) → 범위 밖 경고 |
| `default` | 입력칸 초깃값(중앙값) |
| `mae`, `importance` | 화면에 보여 줄 근거 |

> 변수 이름: `bundle`

In [ ]:
# 힌트: 사전 {"이름": 값, ...} 을 joblib.dump(사전, "파일명")

## 3. 저장한 모델 다시 불러와 확인 ✍️

앱이 하는 일과 똑같이, 파일에서 불러와 예측해 봅니다.

In [ ]:
# 힌트: joblib.load("파일명")

## 4. 예측 앱 만들기

`%%writefile`로 `app_concrete.py`를 세 부분으로 나눠 만듭니다.

**① 앱 틀과 입력칸 📋** 배합 8개를 사이드바에서 입력받습니다. 입력칸은 범위를 막지 않아서 학습 범위 밖 값도 넣어 볼 수 있습니다.

In [ ]:
%%writefile app_concrete.py
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="콘크리트 강도 예측")
st.title("콘크리트 압축강도 예측")

bundle = joblib.load("concrete_model.joblib")
ranges = bundle["ranges"]

st.sidebar.header("배합 조건 (kg/m³, 재령은 일)")
inputs = {}
for col in bundle["columns"]:
    inputs[col] = st.sidebar.number_input(col, value=float(bundle["default"][col]))

**② 예측과 범위 경고 ✍️**

- 입력 사전을 한 줄짜리 표로 바꿔 예측합니다.
- 학습 범위 밖 입력이 하나라도 있으면 `st.warning`을 띄웁니다.

In [ ]:
%%writefile -a app_concrete.py
# 힌트: pd.DataFrame([입력사전]), st.metric("이름", 값), st.warning("문구")

**③ 근거 표시 📋** 모델의 평균 오차와 변수 중요도를 함께 보여 줍니다.

In [ ]:
%%writefile -a app_concrete.py

st.caption(f"이 모델은 평균 {bundle['mae']:.1f} MPa 정도 틀립니다 (Test MAE).")
st.subheader("변수 중요도")
st.bar_chart(bundle["importance"].sort_values(ascending=False))

## 5. 앱 실행 (터미널)

```bash
conda activate ml_l3
cd work/student/day4      # 이 노트북과 concrete_model.joblib이 있는 폴더
streamlit run app_concrete.py
```

시멘트를 600처럼 학습 범위(최대 540) 밖으로 바꿔 경고가 뜨는지 확인해 보세요. 끝낼 때는 `Ctrl + C`.

## 정리 💬

**Q1.** 모델 파일만 저장하고 `ranges`, `columns`는 저장하지 않았다면 어떤 문제가 생길 수 있을까요?

> 답:


**Q2.** 이 앱을 1일차 모니터링 앱과 합친다면 어떤 화면 구성이 좋을까요?

> 답:
